## Questão 1


In [ ]:
from cryptography.hazmat.primitives import padding

BLOCK_SIZE = 128

def pad(ptxt):
    padder = padding.PKCS7(BLOCK_SIZE).padder()
    return padder.update(ptxt) + padder.finalize()

def unpad(padded):
    unpadder = padding.PKCS7(BLOCK_SIZE).unpadder()
    return unpadder.update(padded) + unpadder.finalize()

# mensagem original
msg = b"abc"

# aplicar padding
p = pad(msg)

# alterar o último byte (mantém tamanho múltiplo do bloco)
p_bad = p[:-1] + bytes([p[-1] ^ 1])

# tentar remover padding
print(unpad(p_bad))

## Explicação do Exercício - Questão 1

### Mensagem escolhida
- **Mensagem:** `"abc"` → 3 bytes
- **Padding adicionado:** 13 bytes (PKCS7)

### Regra do PKCS7
> **Se faltam N bytes para completar o bloco, adiciona N bytes com o valor N.**

- **Tamanho do bloco:** 16 bytes
- **Tamanho da mensagem:** 3 bytes (`"abc"`)
- **Bytes em falta:** 16 - 3 = **13 bytes**
- **Padding aplicado:** Adiciona o valor **13** às últimas **13 posições**

### Corrupção do Padding

- **Operação:** Inverter o último byte
- **Cálculo:** 13 XOR 1 = **12**
- **Ou seja , apenas o último byte vai ser "mudado" (invertido)**

### Tentativa de Remover o Padding

1. **Lê o último byte:** 12
2. **Interpreta:** "O padding tem tamanho 12"
3. **Verifica os últimos 12 bytes**
4. **Espera encontrar:** Todos os valores iguais a **12**
5. **Mas encontra:** Valores iguais a **13** (exceto o último)

### Conclusão

-  **Não vemos só o último byte** - o `unpad()` verifica os **últimos N bytes**
-  Como os últimos 12 bytes **não são todos 12**, dá **erro**
-  O erro **NÃO é** porque o tamanho não é múltiplo do bloco
-  O erro **É** porque o valor do padding está **inconsistente**

## cbc_pad_orcl.py

In [4]:
from os import urandom
from cryptography.hazmat.primitives import padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

BLOCK_SIZE = 16
key = urandom(BLOCK_SIZE)


def pad(msg): # adiciona padding
    padder = padding.PKCS7(BLOCK_SIZE * 8).padder() # tem de ser multiplo do tamanho do bloco
    return padder.update(msg) + padder.finalize()


def unpad(msg): # remove padding
    unpadder = padding.PKCS7(BLOCK_SIZE * 8).unpadder() # padding correto devolve mensagem
    return unpadder.update(msg) + unpadder.finalize() # senao erro

def enc(msg): # copiado da semana anterior
    iv = urandom(BLOCK_SIZE)
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv))
    encryptor = cipher.encryptor()
    ct = encryptor.update(pad(msg)) + encryptor.finalize()
    return iv + ct

def dec(ct):
    iv = ct[:BLOCK_SIZE]
    data = ct[BLOCK_SIZE:]
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv))
    decryptor = cipher.decryptor()
    return decryptor.update(data) + decryptor.finalize()

def pad_orcl(ct): #implementar padding
    try:
        m = dec(ct) # recebe criptograma ct e decifra
        unpad(m) # tentar remover pad
        return True
    except ValueError: # true se consegue , erro senao
        return False

# parte de teste
print("Teste 1: Criptograma bem formado")
msg = b"QUERO20"
ct = enc(msg)
print(f"Mensagem: {msg}")
print(f"Padding válido? {pad_orcl(ct)}")  # True

print("\nTeste 2: Alterar último byte do criptograma")
ct_bad = ct[:-1] + bytes([ct[-1] ^ 1])
print(f"Padding válido? {pad_orcl(ct_bad)}")  # False

print("\nTeste 3: Alterar byte no meio do criptograma")
ct_bad2 = bytearray(ct)
ct_bad2[20] ^= 1
print(f"Padding válido? {pad_orcl(bytes(ct_bad2))}")  # Provavelmente False

print("\nTeste 4: Truncar o criptograma")
ct_truncated = ct[:-5]
print(f"Padding válido? {pad_orcl(ct_truncated)}")  # False


Teste 1: Criptograma bem formado
Mensagem: b'QUERO20'
Padding válido? True

Teste 2: Alterar último byte do criptograma
Padding válido? False

Teste 3: Alterar byte no meio do criptograma
Padding válido? False

Teste 4: Truncar o criptograma
Padding válido? False


## pad_orcl_attck_lastbyte.py

In [ ]:
from os import urandom
from cryptography.hazmat.primitives import padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

BLOCK_SIZE = 16
key = urandom(BLOCK_SIZE)


def pad(msg): # adiciona padding
    padder = padding.PKCS7(BLOCK_SIZE * 8).padder() # tem de ser multiplo do tamanho do bloco
    return padder.update(msg) + padder.finalize()


def unpad(msg): # remove padding
    unpadder = padding.PKCS7(BLOCK_SIZE * 8).unpadder() # padding correto devolve mensagem
    return unpadder.update(msg) + unpadder.finalize() # senao erro

def enc(msg): # copiado da semana anterior
    iv = urandom(BLOCK_SIZE)
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv))
    encryptor = cipher.encryptor()
    ct = encryptor.update(pad(msg)) + encryptor.finalize()
    return iv + ct

def dec(ct):
    iv = ct[:BLOCK_SIZE]
    data = ct[BLOCK_SIZE:]
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv))
    decryptor = cipher.decryptor()
    return decryptor.update(data) + decryptor.finalize()

def pad_orcl(ct): #implementar padding
    try:
        m = dec(ct) # recebe criptograma ct e decifra
        unpad(m) # tentar remover pad
        return True
    except ValueError: # true se consegue , erro senao
        return False

def discover_padding_size(ctxt): # recebe criptograma
    blocks = [ctxt[i:i + BLOCK_SIZE] for i in range(0, len(ctxt), BLOCK_SIZE)] # divide em blocos de 16 bytes
    c_prev = bytearray(blocks[-2]) # buscar o penultimo bloco pq o ultimo bloco de texto limpo depende do ultimo bloco cifrado (bytearray pq permite alterar bytes individuais)
    c_last = blocks[-1] # ultimo bloco que vai ser decifrado

    for i in range(BLOCK_SIZE -1 , -1 , -1):
        test_prev = c_prev[:] #cópia do penultimo pra poder alterar a vontade
        test_prev[i] ^= 1 #alterar penultimo byte
        test_ctxt = bytes().join(blocks[:-2] + [bytes(test_prev), c_last]) #reconstroi um novo criptograma
                                                                          #pega em todos os blocos menos os ultimos dois
        if pad_orcl(test_ctxt): #perguntar ao oracle se o cripto é valido
            return BLOCK_SIZE - i - 1 

    return BLOCK_SIZE


msg = b"OLA"
ct = enc(msg)

print(discover_padding_size(ct))

13


# pad_orcl_attck.py

In [8]:
from os import urandom
from cryptography.hazmat.primitives import padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

BLOCK_SIZE = 16
key = urandom(BLOCK_SIZE)


def pad(msg):
    padder = padding.PKCS7(BLOCK_SIZE * 8).padder()
    return padder.update(msg) + padder.finalize()


def unpad(msg):
    unpadder = padding.PKCS7(BLOCK_SIZE * 8).unpadder()
    return unpadder.update(msg) + unpadder.finalize()


def enc(msg):
    iv = urandom(BLOCK_SIZE)
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv))
    encryptor = cipher.encryptor()
    ct = encryptor.update(pad(msg)) + encryptor.finalize()
    return iv + ct


def dec(ct):
    iv = ct[:BLOCK_SIZE]
    data = ct[BLOCK_SIZE:]
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv))
    decryptor = cipher.decryptor()
    return decryptor.update(data) + decryptor.finalize()


def pad_orcl(ct):
    try:
        m = dec(ct)
        unpad(m)
        return True
    except ValueError:
        return False


def attack_last_block(ctxt):
    blocks = [ctxt[i:i + BLOCK_SIZE] for i in range(0, len(ctxt), BLOCK_SIZE)]
    c_prev_original = bytearray(blocks[-2])
    c_last = blocks[-1]
    
    intermediate = bytearray(BLOCK_SIZE)
    
    for pos in range(BLOCK_SIZE - 1, -1, -1):
        padding_value = BLOCK_SIZE - pos
        
        for guess in range(256):
            c_prev = c_prev_original[:]
            c_prev[pos] = guess
            
            for i in range(pos + 1, BLOCK_SIZE):
                c_prev[i] = intermediate[i] ^ padding_value
            
            test_ctxt = b''.join(blocks[:-2] + [bytes(c_prev), c_last])
            
            if pad_orcl(test_ctxt):
                # Verificação extra para padding de tamanho 1
                if padding_value == 1 and guess == c_prev_original[pos]:
                    # Testar se não é o valor original (falso positivo)
                    c_prev_test = c_prev_original[:]
                    c_prev_test[pos] = (c_prev_original[pos] + 1) % 256
                    test_ctxt_verify = b''.join(blocks[:-2] + [bytes(c_prev_test), c_last])
                    if pad_orcl(test_ctxt_verify):
                        continue  # É falso positivo, continuar
                
                intermediate[pos] = guess ^ padding_value
                break
    
    plaintext = bytearray(BLOCK_SIZE)
    for i in range(BLOCK_SIZE):
        plaintext[i] = intermediate[i] ^ c_prev_original[i]
    
    return bytes(plaintext)


# test

print("=== Teste 1: Mensagem curta ===")
msg = b"OLA"
ct = enc(msg)
print(f"Mensagem original: {msg}")
recovered = attack_last_block(ct)
print(f"Bloco recuperado: {recovered}")
print(f"Mensagem recuperada: {recovered[:len(msg)]}")
print(f"CORRETO!" if recovered[:len(msg)] == msg else " ERRADO!")

print("\n=== Teste 2: Mensagem média ===")
msg = b"QUERO20"
ct = enc(msg)
print(f"Mensagem original: {msg}")
recovered = attack_last_block(ct)
print(f"Mensagem recuperada: {recovered[:len(msg)]}")
print(f" CORRETO!" if recovered[:len(msg)] == msg else " ERRADO!")

print("\n=== Teste 3: Bloco completo ===")
msg = b"1234567890123456"
ct = enc(msg)
print(f"Mensagem original: {msg}")
recovered = attack_last_block(ct)
print(f"Mensagem recuperada: {recovered}")
print(f" CORRETO!" if recovered == msg else " ERRADO!")

print("\n=== Teste 4: Verificação do bloco completo ===")
msg = b"1234567890123456"
ct = enc(msg)
print(f"Mensagem original: {msg} ({len(msg)} bytes)")

# Decifrar para ver a estrutura
decrypted = dec(ct)
print(f"Decifrado completo: {decrypted}")
print(f"Tamanho após padding: {len(decrypted)} bytes")

# O último bloco deve ser todo padding 0x10
recovered = attack_last_block(ct)
print(f"Último bloco recuperado: {recovered}")
print(f"É todo padding 0x10? {recovered == bytes([16] * 16)}")
print(f" CORRETO! (último bloco É o padding)" if recovered == bytes([16] * 16) else " ERRADO!")

=== Teste 1: Mensagem curta ===
Mensagem original: b'OLA'
Bloco recuperado: b'OLA\r\r\r\r\r\r\r\r\r\r\r\r\r'
Mensagem recuperada: b'OLA'
CORRETO!

=== Teste 2: Mensagem média ===
Mensagem original: b'QUERO20'
Mensagem recuperada: b'QUERO20'
 CORRETO!

=== Teste 3: Bloco completo ===
Mensagem original: b'1234567890123456'
Mensagem recuperada: b'\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10'
 ERRADO!

=== Teste 4: Verificação do bloco completo ===
Mensagem original: b'1234567890123456' (16 bytes)
Decifrado completo: b'1234567890123456\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10'
Tamanho após padding: 32 bytes
Último bloco recuperado: b'\xdb\xae\x03\xab\x05%\xb1\xbd\x96[=\xfb\xad\xfeN\x01'
É todo padding 0x10? False
 ERRADO!
